# Three things Rust does that Python does not

Pick the **Rust** kernel (**Select Another Kernel... → Jupyter Kernel... → Rust**). If `let` is a `SyntaxError`, you are still on Python.
Polars was faster in part 1 because it is written in Rust. You will not write Rust for homework — just run these cells and read what they did.
Three ideas: the compiler checks a cell before it runs, `let` values do not change unless you write `mut`, and `b = a` hands the value over so `a` is gone.
Two cells are **supposed to fail**. Read the error, then keep going.
Sections 1–3 are the class. Everything after the takeaway is optional.


## 1. Hello

Run this cell. Type Rust the same way you typed Python in part 1: no `fn main()`, just the lines you want to run.


In [55]:
// `let` names a value. It stays put unless you later write `mut`.
let engine = "Rust";
// `println!` prints a line. `{engine}` inserts that value into the text.
println!("Hello from {engine}.");
println!("This cell was checked by the compiler before it ran.");


Hello from Rust.
This cell was checked by the compiler before it ran.


The kernel compiled that cell, then ran it. A typo would have stopped it before anything printed — Python does not do that step.
`pip install polars` downloads this kind of compiled Rust, with a thin Python layer on top.


## 2. Immutable unless you ask

In Python a name can be rebound whenever you like. In Rust, `let` holds still.
Write `let mut` when the value should change — here, a rating cutoff that stays put, and a running row count that does not.


In [56]:
// No `mut`: this cutoff is not allowed to change.
let min_ratings = 1000;
println!("keep movies with at least {min_ratings} ratings");

// `mut` means "this one is allowed to change" — here, a running count.
let mut rows_read = 0;
for _ in 0..3 { // repeat 3 times. `_` means we do not use the loop counter.
    rows_read += 100_000; // add 100,000 more rows each pass
    println!("rows_read = {rows_read}");
}


keep movies with at least 1000 ratings
rows_read = 100000
rows_read = 200000
rows_read = 300000


()

This cell is **supposed to fail**. It tries to change `min_ratings` after a plain `let`.
Read the error: it names both lines and tells you the fix (`mut`).


In [57]:
// This cell is supposed to fail. Read the error, then keep going.
let min_ratings = 1000;
println!("keep movies with at least {min_ratings} ratings");

min_ratings = 2000; // not allowed: we never said `mut`
println!("keep movies with at least {min_ratings} ratings");


Error: cannot assign twice to immutable variable `min_ratings`

In Python the same four lines just work: `min_ratings = 1000` and then `min_ratings = 2000`.
Nothing there was wrong, and that is the problem: Python never asked which values were supposed to hold still.


## 3. Ownership: `b = a` hands the value over

Every value has one owner. `let moved = ratings` *moves* the list: `moved` owns it, and `ratings` is done.
In Python both names would share one list. Rust makes you choose: move it, or `.clone()` it.


In [58]:
// `vec!` is a list. These are three movie ratings from part 1.
let ratings = vec![4.5, 3.0, 5.0];
println!("ratings = {ratings:?}"); // `:?` prints the list in a readable way


ratings = [4.5, 3.0, 5.0]


This cell is **supposed to fail**. It moves the list, then tries to print both names.
Read the error: `ratings` was moved.


In [59]:
// This cell is supposed to fail. Read the error, then keep going.
let ratings = vec![4.5, 3.0, 5.0];
let moved = ratings; // ownership moves: `ratings` is now empty-handed
println!("{ratings:?} {moved:?}"); // using `ratings` here is the error


Error: borrow of moved value: `ratings`

If you really want two copies, say so. `.clone()` is visible in the source — that is the point.


In [60]:
let ratings = vec![4.5, 3.0, 5.0];
let copy = ratings.clone(); // make a second list so both names can live
let moved = ratings; // this move is fine: we still have `copy`
println!("moved = {moved:?}");
println!("copy  = {copy:?}");


moved = [4.5, 3.0, 5.0]
copy  = [4.5, 3.0, 5.0]


## What to take away

Mistakes: Python finds them when the line runs; Rust finds them before the cell runs.
Variables: Python always rebinds; Rust is immutable unless you write `mut`.
`b = a`: Python is a second name for one object; Rust moves ownership and `a` is done.
In one sentence: Rust checks those promises before anything runs — a large part of why Polars can be fast from Python.


## Optional: go further


### A. Where the memory goes

Rust has no garbage collector. The compiler inserts `free` at compile time.
`Drop` prints at the moment a value is freed, so you can watch it.
Look for `FREE 64 MB` *before* the line "back in the outer block".


In [61]:
// A Buffer is a block of bytes we can watch being freed.
struct Buffer {
    data: Vec<u8>, // the actual bytes
}

fn make_buffer(megabytes: usize) -> Buffer {
    println!("  allocate {megabytes} MB");
    Buffer {
        data: vec![0; megabytes * 1024 * 1024],
    }
}

// `Drop` runs automatically when a Buffer goes out of scope — no `free()` call.
impl Drop for Buffer {
    fn drop(&mut self) {
        let megabytes = self.data.len() / 1024 / 1024;
        println!("  FREE {megabytes} MB"); // this prints at the moment of free
    }
}

println!("enter outer block");
{
    println!("  enter inner scope");
    let _scratch = make_buffer(64); // create it only to watch it die
    println!("  inner scope is about to end");
} // scratch is freed on this brace
println!("back in the outer block: that memory is already gone");


enter outer block
  enter inner scope
  allocate 64 MB
  inner scope is about to end
  FREE 64 MB
back in the outer block: that memory is already gone


CPython frees when the last name is gone, by counting references at runtime.
Counting cannot free a cycle (two objects that point at each other), so Python also has a cycle collector.
In Rust you do not stumble into a cycle: shared ownership is a type you ask for by name (`Rc`).


### B. Borrowing: one writer or many readers

Moving every time would be exhausting, so you can *lend* a value instead.
`&T` is shared and read-only (as many as you like). `&mut T` is exclusive and may write (exactly one).
One writer, or many readers — never both. That is the borrow checker.


In [62]:
let mut ratings = vec![5, 3, 4, 4];
{
    // `&` lends the list for reading. Two readers at once is allowed.
    let first = &ratings;
    let second = &ratings;
    println!("two readers: {first:?} and {second:?}");
} // the loans end here, so a write is allowed again
ratings.push(2); // add a 2-star rating
println!("after a write: {ratings:?}");


two readers: [5, 3, 4, 4] and [5, 3, 4, 4]
after a write: [5, 3, 4, 4, 2]


The obvious bug — change the list while you loop over it — does not compile.
This cell is **supposed to fail**.


In [63]:
// This cell is supposed to fail: change the list while looping over it.
let mut ratings = vec![5, 3, 4];
for r in &ratings { // `r` is a reader (a loan)
    if *r == 3 { // `*r` is the number the loan points at
        ratings.push(0); // write while the reader is still alive — not allowed
    }
}


Error: cannot borrow `ratings` as mutable because it is also borrowed as immutable

In Python the same idea runs and gives the wrong answer.
Looping `ratings.remove(rating)` on `[5, 2, 2, 4]` leaves `[5, 2, 4]`: one `2` survived, and there was no error.


### C. Threads vs the GIL

CPython lets only one thread run Python bytecode at a time (the GIL), so extra threads do not speed up a tight loop.
Rust has no such lock. Each thread below gets its own slice of the work.
The `total` column should stay the same; `speedup` should climb.


In [64]:
use std::thread;
use std::time::Instant;

// Add up one slice of the work. `%` is remainder; it just keeps the number small.
fn partial_sum(from: u64, to: u64) -> u64 {
    let mut total = 0;
    for i in from..to {
        total += (i * i) % 1_000_003;
    }
    total
}

let iterations: u64 = 80_000_000; // how many numbers to process in total
println!("{iterations} iterations\n");
println!("threads   seconds   speedup");

let mut one_thread = 0.0; // save the 1-thread time so we can print speedup
for threads in [1u64, 2, 4, 8] {
    let start = Instant::now(); // stopwatch on
    let chunk = iterations / threads; // each thread gets this many numbers
    let total = thread::scope(|scope| {
        let mut handles = Vec::new(); // tickets we wait on later
        for t in 0..threads {
            let from = t * chunk;
            let mut to = from + chunk;
            if t == threads - 1 {
                to = iterations; // last thread takes any leftover
            }
            // `move` gives this thread its own copy of `from` and `to`
            handles.push(scope.spawn(move || partial_sum(from, to)));
        }
        let mut total = 0;
        for handle in handles {
            total += handle.join().unwrap(); // wait, then add this thread's piece
        }
        total
    });
    let seconds = start.elapsed().as_secs_f64(); // stopwatch off
    if threads == 1 {
        one_thread = seconds;
    }
    // speedup = (1-thread time) / (this time). 2.00x means twice as fast.
    println!("{threads:>7}   {seconds:>7.3}   {:>6.2}x   (checksum {total})", one_thread / seconds);
}


80000000 iterations

threads   seconds   speedup
      1     0.047     1.00x   (checksum 39991795338200)
      2     0.021     2.26x   (checksum 39991795338200)
      4     0.011     4.19x   (checksum 39991795338200)
      8     0.006     7.74x   (checksum 39991795338200)


()

Python's line would be flat (~1.0× at 8 threads). Extra threads spend their time handing the GIL back and forth.
Polars releases that lock while its compiled Rust runs, which is why it can use all your cores from a Python process.


### D. A data race is a compile error

The borrow rule also makes threads safe: two threads cannot both write to one counter.
This cell is **supposed to fail**.


In [65]:
// This cell is supposed to fail: four threads all try to write `total` at once.
let mut total = 0u64;
std::thread::scope(|scope| {
    for _ in 0..4 {
        scope.spawn(|| {
            total += 1; // two writers at once is a data race
        });
    }
});


Error: cannot borrow `total` as mutable more than once at a time

In Python the same bug often looks like it works, because the GIL happens to serialize the increments.
Here it never becomes a program you can run.


### E. Debug vs release, and why Polars is fast

`:opt 0` is a debug build; `:opt 3` is release. Same source; the compiler just works harder.
Both cells run the **same query as part 1** on `data/formats/ratings_1m.csv` (one million MovieLens ratings).
The ranking must match (movie 318 first at about 4.375). Only the time should change.
Polars from Python is the second number: a release Rust library, which is why a Python one-liner can beat a Python `for` loop.


In [60]:
:opt 0


Optimization: 0


In [61]:
// Same query as part 1: group ratings by movie, keep popular ones, print the top 10.

use std::collections::HashMap;
use std::fs;
use std::path::PathBuf;
use std::time::Instant;

// Find the CSV whether Jupyter started in the repo root or in `notebooks/`.
let mut path = PathBuf::from("data/formats/ratings_1m.csv");
if !path.exists() {
    path = PathBuf::from("../data/formats/ratings_1m.csv");
}
// One million lines into one string. (Not timed — this is just reading the disk.)
let text = fs::read_to_string(&path).expect("run part 1 first so data/formats/ratings_1m.csv exists");

let start = Instant::now(); // stopwatch on — we time the query only

// Two dictionaries: movie → sum of stars, and movie → how many ratings.
let mut totals: HashMap<i32, f64> = HashMap::new();
let mut counts: HashMap<i32, u32> = HashMap::new();
let mut rows = 0u32;

for line in text.lines().skip(1) { // skip the header: userId,movieId,rating,...
    let mut fields = line.split(','); // columns, left to right
    let _user = fields.next(); // we do not need the user id
    let movie: i32 = fields.next().unwrap().parse().unwrap(); // column 2 (whole number)
    let rating: f64 = fields.next().unwrap().parse().unwrap(); // column 3 (decimal)
    *totals.entry(movie).or_insert(0.0) += rating; // add this rating to the movie's total
    *counts.entry(movie).or_insert(0) += 1; // count +1 for this movie
    rows += 1;
}

let min_ratings = 1000; // same cutoff as part 1
let mut ranked = Vec::new();
for (movie, n) in &counts {
    if *n < min_ratings {
        continue; // skip movies that are too rare for a fair average
    }
    let average = totals[movie] / f64::from(*n); // mean rating
    ranked.push((average, *n, *movie));
}
// Highest average first; if tied, smaller movie id first (same as part 1).
ranked.sort_by(|a, b| b.0.partial_cmp(&a.0).unwrap().then(a.2.cmp(&b.2)));
let seconds = start.elapsed().as_secs_f64(); // stopwatch off

println!("read {rows} ratings from {}", path.display());
println!("{} movies with at least {min_ratings} ratings, {seconds:.3} s\n", ranked.len());
println!("  movie      n  average");
for (average, n, movie) in ranked.iter().take(10) { // only the ten highest
    println!("{movie:>7}  {n:>5}  {average:.3}");
}


read 1000000 ratings from data/formats/ratings_1m.csv
126 movies with at least 1000 ratings, 0.412 s

  movie      n  average
    318   3212  4.375
    858   2000  4.311
   1221   1332  4.268
    527   2337  4.267
     50   2068  4.242
   5618   1041  4.230
   1193   1371  4.222
    750   1008  4.210
   2959   2393  4.199
    296   2989  4.198


()


In [62]:
:opt 3


Optimization: 3


In [63]:
// Same query as part 1: group ratings by movie, keep popular ones, print the top 10.

use std::collections::HashMap;
use std::fs;
use std::path::PathBuf;
use std::time::Instant;

// Find the CSV whether Jupyter started in the repo root or in `notebooks/`.
let mut path = PathBuf::from("data/formats/ratings_1m.csv");
if !path.exists() {
    path = PathBuf::from("../data/formats/ratings_1m.csv");
}
// One million lines into one string. (Not timed — this is just reading the disk.)
let text = fs::read_to_string(&path).expect("run part 1 first so data/formats/ratings_1m.csv exists");

let start = Instant::now(); // stopwatch on — we time the query only

// Two dictionaries: movie → sum of stars, and movie → how many ratings.
let mut totals: HashMap<i32, f64> = HashMap::new();
let mut counts: HashMap<i32, u32> = HashMap::new();
let mut rows = 0u32;

for line in text.lines().skip(1) { // skip the header: userId,movieId,rating,...
    let mut fields = line.split(','); // columns, left to right
    let _user = fields.next(); // we do not need the user id
    let movie: i32 = fields.next().unwrap().parse().unwrap(); // column 2 (whole number)
    let rating: f64 = fields.next().unwrap().parse().unwrap(); // column 3 (decimal)
    *totals.entry(movie).or_insert(0.0) += rating; // add this rating to the movie's total
    *counts.entry(movie).or_insert(0) += 1; // count +1 for this movie
    rows += 1;
}

let min_ratings = 1000; // same cutoff as part 1
let mut ranked = Vec::new();
for (movie, n) in &counts {
    if *n < min_ratings {
        continue; // skip movies that are too rare for a fair average
    }
    let average = totals[movie] / f64::from(*n); // mean rating
    ranked.push((average, *n, *movie));
}
// Highest average first; if tied, smaller movie id first (same as part 1).
ranked.sort_by(|a, b| b.0.partial_cmp(&a.0).unwrap().then(a.2.cmp(&b.2)));
let seconds = start.elapsed().as_secs_f64(); // stopwatch off

println!("read {rows} ratings from {}", path.display());
println!("{} movies with at least {min_ratings} ratings, {seconds:.3} s\n", ranked.len());
println!("  movie      n  average");
for (average, n, movie) in ranked.iter().take(10) { // only the ten highest
    println!("{movie:>7}  {n:>5}  {average:.3}");
}


read 1000000 ratings from data/formats/ratings_1m.csv
126 movies with at least 1000 ratings, 0.059 s

  movie      n  average
    318   3212  4.375
    858   2000  4.311
   1221   1332  4.268
    527   2337  4.267
     50   2068  4.242
   5618   1041  4.230
   1193   1371  4.222
    750   1008  4.210
   2959   2393  4.199
    296   2989  4.198


()


Same query, same ranking. Only the time dropped.
That faster number is what Polars is doing under the hood when you call it from Python.
Run `movielens_dataframe_engines_simple.ipynb` first if `data/formats/ratings_1m.csv` is missing.
